In [1]:
# ============================================================
# V10 — CELL 1
# IMPORTS + PROJECT CONFIGURATION
# ============================================================

import os
import glob
import time
import warnings

import numpy as np
import pandas as pd

# Geospatial / raster
import rasterio
from rasterio.windows import Window

# Spatial ML / nearest-neighbour
from sklearn.neighbors import BallTree

# Machine learning
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Pandas display configuration
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:.4f}"
)


# ------------------------------------------------------------
# Geographic constants
# ------------------------------------------------------------

EARTH_RADIUS_KM = 6371.0088

# WorldCover resolution
WORLDCOVER_RESOLUTION_M = 10

# Context radii
RADIUS_300M = 300
RADIUS_1KM = 1000
RADIUS_3KM = 3000


# ------------------------------------------------------------
# Project files
# ------------------------------------------------------------

VIIRS_EVENT_FILE = (
    "viirs_poc_detection_event_mapping.csv"
)

OSM_FILE = (
    "osm_points_v7.csv"
)

WORLD_COVER_DIR = (
    "worldcover_india"
)


# ------------------------------------------------------------
# Output files
# ------------------------------------------------------------

V10_FEATURE_FILE = (
    "viirs_event_features_v10.csv"
)

V10_CLUSTER_FILE = (
    "viirs_event_behavior_clusters_v10.csv"
)

V10_FINAL_FILE = (
    "viirs_event_final_classification_v10.csv"
)


# ------------------------------------------------------------
# WorldCover classes
# ------------------------------------------------------------

WORLDCOVER_CLASSES = {
    10: "Tree cover",
    20: "Shrubland",
    30: "Grassland",
    40: "Cropland",
    50: "Built-up",
    60: "Bare/sparse vegetation",
    70: "Snow/ice",
    80: "Permanent water",
    90: "Herbaceous wetland",
    95: "Mangroves",
    100: "Moss/lichen"
}


# ------------------------------------------------------------
# Project status
# ------------------------------------------------------------

print("=" * 70)
print("SIH — INDUSTRIAL THERMAL SOURCE DETECTION")
print("V10 CLEAN PIPELINE")
print("=" * 70)

print("\nLibraries loaded.")
print("Configuration loaded.")

print("\nProject files:")
print(f"VIIRS event mapping : {VIIRS_EVENT_FILE}")
print(f"OSM context         : {OSM_FILE}")
print(f"WorldCover directory: {WORLD_COVER_DIR}")

print("\nContext radii:")
print(f"300 m : {RADIUS_300M} m")
print(f"1 km  : {RADIUS_1KM} m")
print(f"3 km  : {RADIUS_3KM} m")

print("\nWorldCover classes:")
print(len(WORLDCOVER_CLASSES))

SIH — INDUSTRIAL THERMAL SOURCE DETECTION
V10 CLEAN PIPELINE

Libraries loaded.
Configuration loaded.

Project files:
VIIRS event mapping : viirs_poc_detection_event_mapping.csv
OSM context         : osm_points_v7.csv
WorldCover directory: worldcover_india

Context radii:
300 m : 300 m
1 km  : 1000 m
3 km  : 3000 m

WorldCover classes:
11


In [2]:
# ============================================================
# V10 — CELL 2
# LOAD CORE DATASETS
# ============================================================

print("=" * 70)
print("LOADING CORE DATASETS")
print("=" * 70)


# ------------------------------------------------------------
# 1. VIIRS event mapping
# ------------------------------------------------------------

if not os.path.exists(VIIRS_EVENT_FILE):
    raise FileNotFoundError(
        f"Missing file: {VIIRS_EVENT_FILE}"
    )

viirs = pd.read_csv(
    VIIRS_EVENT_FILE,
    parse_dates=["acq_date"]
)

print("\nVIIRS EVENT MAPPING")
print("-" * 70)

print(f"Rows    : {len(viirs):,}")
print(f"Columns : {len(viirs.columns)}")

print("\nColumns:")
print(
    viirs.columns.tolist()
)


# ------------------------------------------------------------
# 2. OSM context
# ------------------------------------------------------------

if not os.path.exists(OSM_FILE):
    raise FileNotFoundError(
        f"Missing file: {OSM_FILE}"
    )

osm = pd.read_csv(
    OSM_FILE
)

print("\nOSM CONTEXT")
print("-" * 70)

print(f"Rows    : {len(osm):,}")
print(f"Columns : {len(osm.columns)}")

print("\nColumns:")
print(
    osm.columns.tolist()
)


# ------------------------------------------------------------
# 3. WorldCover files
# ------------------------------------------------------------

worldcover_files = sorted(
    glob.glob(
        os.path.join(
            WORLD_COVER_DIR,
            "ESA_WorldCover_10m_2021_v200_*_Map.tif"
        )
    )
)

if not worldcover_files:
    raise FileNotFoundError(
        f"No WorldCover TIFFs found in "
        f"{WORLD_COVER_DIR}"
    )

print("\nWORLDCOVER")
print("-" * 70)

print(
    f"TIFF files found : "
    f"{len(worldcover_files)}"
)


# ------------------------------------------------------------
# 4. Basic validation
# ------------------------------------------------------------

print("\nBASIC VALIDATION")
print("-" * 70)

print(
    f"VIIRS missing event_id : "
    f"{viirs['event_id'].isna().sum()}"
)

print(
    f"VIIRS duplicate rows   : "
    f"{viirs.duplicated().sum()}"
)

print(
    f"OSM missing latitude   : "
    f"{osm['latitude'].isna().sum()}"
)

print(
    f"OSM missing longitude  : "
    f"{osm['longitude'].isna().sum()}"
)


# ------------------------------------------------------------
# 5. Event count
# ------------------------------------------------------------

event_count = (
    viirs["event_id"]
    .nunique()
)

print(
    f"\nUnique VIIRS events : "
    f"{event_count:,}"
)

print("\nCore datasets loaded successfully.")

LOADING CORE DATASETS

VIIRS EVENT MAPPING
----------------------------------------------------------------------
Rows    : 13,609
Columns : 8

Columns:
['acq_date', 'daily_object_id', 'event_id', 'latitude', 'longitude', 'frp', 'bright_ti4', 'bright_ti5']

OSM CONTEXT
----------------------------------------------------------------------
Rows    : 1,981
Columns : 15

Columns:
['osm_type', 'osm_id', 'latitude', 'longitude', 'industrial', 'landuse', 'power', 'man_made', 'building', 'product', 'plant_source', 'plant_method', 'resource', 'description', 'source_file']

WORLDCOVER
----------------------------------------------------------------------
TIFF files found : 102

BASIC VALIDATION
----------------------------------------------------------------------
VIIRS missing event_id : 0
VIIRS duplicate rows   : 0
OSM missing latitude   : 0
OSM missing longitude  : 0

Unique VIIRS events : 3,337

Core datasets loaded successfully.


In [3]:
# ============================================================
# V10 — CELL 3
# BUILD EVENT-LEVEL THERMAL / TEMPORAL / SPATIAL FEATURES
# ============================================================

print("=" * 70)
print("BUILDING EVENT-LEVEL FEATURES")
print("=" * 70)


# ------------------------------------------------------------
# 1. Ensure dates are datetime
# ------------------------------------------------------------

viirs["acq_date"] = pd.to_datetime(
    viirs["acq_date"]
)


# ------------------------------------------------------------
# 2. Thermal features
# ------------------------------------------------------------

thermal_features = (
    viirs
    .groupby("event_id")
    .agg(
        mean_frp=("frp", "mean"),
        max_frp=("frp", "max"),
        std_frp=("frp", "std"),

        mean_bright_ti4=(
            "bright_ti4",
            "mean"
        ),

        max_bright_ti4=(
            "bright_ti4",
            "max"
        ),

        std_bright_ti4=(
            "bright_ti4",
            "std"
        ),

        mean_bright_ti5=(
            "bright_ti5",
            "mean"
        ),

        max_bright_ti5=(
            "bright_ti5",
            "max"
        ),

        std_bright_ti5=(
            "bright_ti5",
            "std"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 3. Temporal features
# ------------------------------------------------------------

temporal_features = (
    viirs
    .groupby("event_id")
    .agg(
        start_date=("acq_date", "min"),
        end_date=("acq_date", "max"),
        active_days=("acq_date", "nunique"),
        detection_count=("event_id", "size")
    )
    .reset_index()
)

temporal_features["duration_days"] = (
    (
        temporal_features["end_date"]
        -
        temporal_features["start_date"]
    ).dt.days
    + 1
)

temporal_features["activity_frequency"] = (
    temporal_features["active_days"]
    /
    temporal_features["duration_days"]
)

temporal_features["detections_per_active_day"] = (
    temporal_features["detection_count"]
    /
    temporal_features["active_days"]
)


# ------------------------------------------------------------
# 4. Spatial centroid
# ------------------------------------------------------------

spatial_features = (
    viirs
    .groupby("event_id")
    .agg(
        centroid_lat=("latitude", "mean"),
        centroid_lon=("longitude", "mean")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 5. Haversine distance
# ------------------------------------------------------------

def haversine_km(
    lat1,
    lon1,
    lat2,
    lon2
):

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)

    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        +
        np.cos(lat1)
        *
        np.cos(lat2)
        *
        np.sin(dlon / 2) ** 2
    )

    return (
        EARTH_RADIUS_KM
        *
        2
        *
        np.arcsin(
            np.sqrt(a)
        )
    )


# ------------------------------------------------------------
# 6. Spatial extent
# ------------------------------------------------------------

event_points = viirs.merge(
    spatial_features,
    on="event_id",
    how="left"
)

event_points[
    "distance_from_centroid_km"
] = haversine_km(
    event_points["latitude"],
    event_points["longitude"],
    event_points["centroid_lat"],
    event_points["centroid_lon"]
)

extent_features = (
    event_points
    .groupby("event_id")
    [
        "distance_from_centroid_km"
    ]
    .max()
    .reset_index(
        name="spatial_extent_km"
    )
)


# ------------------------------------------------------------
# 7. Combine
# ------------------------------------------------------------

events = (
    thermal_features
    .merge(
        temporal_features,
        on="event_id"
    )
    .merge(
        spatial_features,
        on="event_id"
    )
    .merge(
        extent_features,
        on="event_id"
    )
)


# ------------------------------------------------------------
# 8. Derived thermal features
# ------------------------------------------------------------

events["frp_range"] = (
    events["max_frp"]
    -
    events["mean_frp"]
)

events["ti4_range"] = (
    events["max_bright_ti4"]
    -
    events["mean_bright_ti4"]
)

events["ti5_range"] = (
    events["max_bright_ti5"]
    -
    events["mean_bright_ti5"]
)


# ------------------------------------------------------------
# 9. Missing standard deviations
# ------------------------------------------------------------

std_columns = [
    "std_frp",
    "std_bright_ti4",
    "std_bright_ti5"
]

events[std_columns] = (
    events[std_columns]
    .fillna(0)
)


# ------------------------------------------------------------
# 10. Diagnostics
# ------------------------------------------------------------

print(
    f"\nEvents generated : "
    f"{len(events):,}"
)

print(
    f"Columns generated: "
    f"{len(events.columns)}"
)

print(
    f"Missing values   : "
    f"{events.isna().sum().sum()}"
)

print("\nEvent feature table:")
display(
    events.head()
)

BUILDING EVENT-LEVEL FEATURES

Events generated : 3,337
Columns generated: 23
Missing values   : 0

Event feature table:


,event_id,mean_frp,max_frp,std_frp,mean_bright_ti4,max_bright_ti4,std_bright_ti4,mean_bright_ti5,max_bright_ti5,std_bright_ti5,start_date,end_date,active_days,detection_count,duration_days,activity_frequency,detections_per_active_day,centroid_lat,centroid_lon,spatial_extent_km,frp_range,ti4_range,ti5_range
0,1,2.0950,2.2100,0.1626,301.0900,304.9900,5.5154,277.2550,277.2700,0.0212,2024-01-01,2024-01-01,1,2,1,1.0000,2.0000,30.5579,79.1196,0.0614,0.1150,3.9000,0.0150
1,2,1.6600,1.6600,0.0000,309.7600,309.7600,0.0000,277.5900,277.5900,0.0000,2024-01-01,2024-01-01,1,1,1,1.0000,1.0000,30.0443,80.5822,0.0000,0.0000,0.0000,0.0000
2,3,0.9727,2.0200,0.3573,305.5497,316.6600,5.6908,282.8742,285.1700,1.9206,2024-01-01,2024-01-29,23,33,29,0.7931,1.4348,27.4695,95.4220,0.3396,1.0473,11.1103,2.2958
3,4,1.9026,4.1900,0.9092,311.6979,328.6800,9.1680,288.7890,292.2800,2.4260,2024-01-01,2024-01-17,17,39,17,1.0000,2.2941,21.7580,83.8424,0.6851,2.2874,16.9821,3.4910
4,5,1.2100,1.2100,0.0000,300.0300,300.0300,0.0000,289.4000,289.4000,0.0000,2024-01-01,2024-01-01,1,1,1,1.0000,1.0000,21.7341,83.9805,0.0000,0.0000,0.0000,0.0000


In [5]:
# CHECK WHAT WORLDCOVER DATA ALREADY EXISTS

print("landcover exists:", "landcover" in globals())

if "landcover" in globals():
    print("landcover shape:", landcover.shape)
    print("landcover columns:", len(landcover.columns))

print("\nExisting CSV files:")
for f in os.listdir("."):
    if "land" in f.lower() or "world" in f.lower():
        print(f)

landcover exists: False

Existing CSV files:
worldcover_india


In [6]:
# ================================================================
# CELL 4 — WORLDCOVER CENTROID CLASS
# ================================================================

print("=" * 70)
print("EXTRACTING WORLDCOVER CENTROID CLASS")
print("=" * 70)

wc_classes = {
    10: "tree",
    20: "shrub",
    30: "grass",
    40: "crop",
    50: "builtup",
    60: "bare",
    70: "snow",
    80: "water",
    90: "wetland",
    95: "mangrove",
    100: "moss"
}

# ------------------------------------------------
# Build tile index once
# ------------------------------------------------

tile_index = []

for tif in worldcover_files:

    try:
        with rasterio.open(tif) as src:

            tile_index.append({
                "path": tif,
                "left": src.bounds.left,
                "right": src.bounds.right,
                "bottom": src.bounds.bottom,
                "top": src.bounds.top
            })

    except Exception:
        pass

tile_index = pd.DataFrame(tile_index)

print(f"Valid WorldCover tiles: {len(tile_index)}")


# ------------------------------------------------
# Assign tile to each event
# ------------------------------------------------

def get_tile_path(lat, lon):

    match = tile_index[
        (tile_index["left"] <= lon) &
        (tile_index["right"] >= lon) &
        (tile_index["bottom"] <= lat) &
        (tile_index["top"] >= lat)
    ]

    if len(match) == 0:
        return None

    return match.iloc[0]["path"]


events["worldcover_tile"] = [
    get_tile_path(lat, lon)
    for lat, lon in zip(
        events["centroid_lat"],
        events["centroid_lon"]
    )
]

print(
    f"Events assigned to tiles: "
    f"{events['worldcover_tile'].notna().sum():,}"
)

print(
    f"Events without tile: "
    f"{events['worldcover_tile'].isna().sum():,}"
)


# ------------------------------------------------
# Extract ONLY centroid pixels
# ------------------------------------------------

records = []

valid_events = events[
    events["worldcover_tile"].notna()
].copy()

groups = valid_events.groupby("worldcover_tile")

start_time = time.time()

for idx, (tile_path, tile_events) in enumerate(
    groups,
    start=1
):

    print(
        f"[{idx:02d}/{len(groups):02d}] "
        f"{os.path.basename(tile_path)} "
        f"→ {len(tile_events)} events"
    )

    try:

        with rasterio.open(tile_path) as src:

            # Read all event coordinates at once
            coords = list(
                zip(
                    tile_events["centroid_lon"],
                    tile_events["centroid_lat"]
                )
            )

            # Sample only one pixel per event
            samples = src.sample(coords)

            for (_, event), sample in zip(
                tile_events.iterrows(),
                samples
            ):

                class_code = int(sample[0])

                record = {
                    "event_id": event["event_id"],
                    "worldcover_class_code": class_code,
                    "worldcover_class": wc_classes.get(
                        class_code,
                        "unknown"
                    )
                }

                records.append(record)

    except Exception as e:

        print(
            f"  WARNING: failed to process "
            f"{os.path.basename(tile_path)}"
        )


# ------------------------------------------------
# Create dataframe
# ------------------------------------------------

landcover = pd.DataFrame(records)

elapsed = time.time() - start_time

print("\n" + "=" * 70)
print("WORLDCOVER EXTRACTION COMPLETE")
print("=" * 70)

print(f"Events processed : {len(landcover):,}")
print(f"Time             : {elapsed:.2f} seconds")

print("\nClass distribution:")

print(
    landcover["worldcover_class"]
    .value_counts()
)

EXTRACTING WORLDCOVER CENTROID CLASS
Valid WorldCover tiles: 102
Events assigned to tiles: 3,337
Events without tile: 0
[01/47] ESA_WorldCover_10m_2021_v200_N06E075_Map.tif → 3 events
[02/47] ESA_WorldCover_10m_2021_v200_N06E078_Map.tif → 3 events
[03/47] ESA_WorldCover_10m_2021_v200_N09E075_Map.tif → 39 events
[04/47] ESA_WorldCover_10m_2021_v200_N09E078_Map.tif → 15 events
[05/47] ESA_WorldCover_10m_2021_v200_N09E090_Map.tif → 1 events
[06/47] ESA_WorldCover_10m_2021_v200_N12E072_Map.tif → 5 events
[07/47] ESA_WorldCover_10m_2021_v200_N12E075_Map.tif → 153 events
[08/47] ESA_WorldCover_10m_2021_v200_N12E078_Map.tif → 53 events
[09/47] ESA_WorldCover_10m_2021_v200_N15E072_Map.tif → 172 events
[10/47] ESA_WorldCover_10m_2021_v200_N15E075_Map.tif → 313 events
[11/47] ESA_WorldCover_10m_2021_v200_N15E078_Map.tif → 63 events
[12/47] ESA_WorldCover_10m_2021_v200_N15E081_Map.tif → 45 events
[13/47] ESA_WorldCover_10m_2021_v200_N18E069_Map.tif → 8 events
[14/47] ESA_WorldCover_10m_2021_v200_

In [7]:
# ================================================================
# CELL 4B — MERGE WORLDCOVER
# ================================================================

# Remove temporary tile assignment
if "worldcover_tile" in events.columns:
    events = events.drop(columns=["worldcover_tile"])

# Remove previous WorldCover columns if present
for col in [
    "worldcover_class_code",
    "worldcover_class"
]:
    if col in events.columns:
        events = events.drop(columns=[col])

# Merge
events = events.merge(
    landcover,
    on="event_id",
    how="left"
)

print("=" * 70)
print("WORLDCOVER MERGED")
print("=" * 70)

print(f"Events : {len(events):,}")

print(
    f"Missing WorldCover: "
    f"{events['worldcover_class'].isna().sum():,}"
)

display(
    events[
        [
            "event_id",
            "centroid_lat",
            "centroid_lon",
            "worldcover_class_code",
            "worldcover_class"
        ]
    ].head(10)
)

WORLDCOVER MERGED
Events : 3,337
Missing WorldCover: 195


,event_id,centroid_lat,centroid_lon,worldcover_class_code,worldcover_class
0,1,30.5579,79.1196,10.0000,tree
1,2,30.0443,80.5822,30.0000,grass
2,3,27.4695,95.4220,10.0000,tree
3,4,21.7580,83.8424,60.0000,bare
4,5,21.7341,83.9805,40.0000,crop
5,6,21.6789,84.0389,60.0000,bare
6,7,21.4870,81.7687,60.0000,bare
7,8,20.9990,86.0126,10.0000,tree
8,9,20.9995,86.0263,50.0000,builtup
9,10,21.0762,85.0394,60.0000,bare


In [8]:
# ================================================================
# CELL 5 — OSM GEOGRAPHIC ENTITY LAYER
# ================================================================

print("=" * 70)
print("BUILDING OSM GEOGRAPHIC ENTITY LAYER")
print("=" * 70)

# ------------------------------------------------
# Inspect available OSM categories
# ------------------------------------------------

print("OSM shape:", osm.shape)

print("\nIndustrial values:")
print(
    osm["industrial"]
    .dropna()
    .value_counts()
    .head(30)
)

print("\nMan-made values:")
print(
    osm["man_made"]
    .dropna()
    .value_counts()
    .head(20)
)

print("\nPower values:")
print(
    osm["power"]
    .dropna()
    .value_counts()
    .head(20)
)

print("\nLanduse values:")
print(
    osm["landuse"]
    .dropna()
    .value_counts()
    .head(20)
)

BUILDING OSM GEOGRAPHIC ENTITY LAYER
OSM shape: (1981, 15)

Industrial values:
industrial
slaughterhouse            115
scrap_yard                 47
depot                      39
Pharmaceutical Company     14
grinding_mill              10
factory                     9
warehouse                   7
concrete_plant              7
brickyard                   6
Pharmaceutical company      3
oil                         3
Biotechnology Company       3
Research Company            2
sawmill                     2
mill                        2
rice_mill                   2
machine_shop                2
oil_mill                    2
food_industry               2
mine                        1
business                    1
Research institute          1
Agrochemical Company        1
Laboratory                  1
rice                        1
distributor                 1
refractory_supplier         1
Name: count, dtype: int64

Man-made values:
man_made
works               1563
wastewater_plant      

In [9]:
# ================================================================
# CELL 5B — OSM ENTITY CLASSIFICATION
# ================================================================

print("=" * 70)
print("CREATING OSM ENTITY CLASSES")
print("=" * 70)

osm = osm.copy()

# ------------------------------------------------
# Normalize text fields
# ------------------------------------------------

for col in [
    "industrial",
    "man_made",
    "power",
    "landuse",
    "product"
]:
    if col in osm.columns:
        osm[col] = (
            osm[col]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.lower()
        )

# ------------------------------------------------
# Define entity classes
# ------------------------------------------------

osm["entity_class"] = "other"

# Industrial zone
osm.loc[
    osm["landuse"].eq("industrial"),
    "entity_class"
] = "industrial_zone"

# Factory / manufacturing
factory_values = {
    "factory",
    "concrete_plant",
    "machine_shop",
    "food_industry",
    "biotechnology company",
    "pharmaceutical company",
    "research company",
    "research institute",
    "laboratory",
    "agrochemical company",
    "refractory_supplier",
    "oil",
    "oil_mill",
    "grinding_mill",
    "sawmill",
    "mill",
    "rice_mill",
    "rice"
}

osm.loc[
    osm["industrial"].isin(factory_values),
    "entity_class"
] = "factory"

# Mines / quarry
osm.loc[
    (
        osm["industrial"].eq("mine")
        |
        osm["landuse"].eq("quarry")
    ),
    "entity_class"
] = "mine"

# Brick-related
brick_values = {
    "brickyard",
    "brickworks"
}

osm.loc[
    osm["industrial"].isin(brick_values),
    "entity_class"
] = "brick_industry"

# Kilns
osm.loc[
    osm["man_made"].eq("kiln"),
    "entity_class"
] = "brick_industry"

# Works
osm.loc[
    osm["man_made"].eq("works"),
    "entity_class"
] = "works"

# Depots
osm.loc[
    osm["industrial"].eq("depot"),
    "entity_class"
] = "depot"

# Power plant
osm.loc[
    osm["power"].eq("plant"),
    "entity_class"
] = "power_plant"

# Other explicit industrial entities
other_industrial = {
    "slaughterhouse",
    "scrap_yard",
    "warehouse",
    "port",
    "cooling",
    "distributor",
    "business"
}

osm.loc[
    osm["industrial"].isin(other_industrial),
    "entity_class"
] = "other_industry"


# ------------------------------------------------
# Summary
# ------------------------------------------------

print("\nEntity class distribution:")
print(
    osm["entity_class"]
    .value_counts()
)

print("\nTotal OSM points:", len(osm))

CREATING OSM ENTITY CLASSES

Entity class distribution:
entity_class
works              1563
other_industry      171
industrial_zone     140
factory              52
depot                39
other                 9
brick_industry        6
mine                  1
Name: count, dtype: int64

Total OSM points: 1981


In [10]:
# ================================================================
# CELL 5C — EVENT → OSM DISTANCE AND DENSITY
# ================================================================

print("=" * 70)
print("CALCULATING EVENT → OSM ENTITY CONTEXT")
print("=" * 70)

# ------------------------------------------------
# Event coordinates
# ------------------------------------------------

event_coords_rad = np.radians(
    events[
        ["centroid_lat", "centroid_lon"]
    ].values
)

# ------------------------------------------------
# Helper function
# ------------------------------------------------

def calculate_entity_context(
    entity_df,
    entity_name,
    event_coords_rad,
    radius_1km=1.0,
    radius_3km=3.0
):

    result = pd.DataFrame(
        index=events.index
    )

    # No entities of this class
    if len(entity_df) == 0:

        result[
            f"distance_to_{entity_name}_km"
        ] = np.inf

        result[
            f"{entity_name}_count_1km"
        ] = 0

        result[
            f"{entity_name}_count_3km"
        ] = 0

        return result

    # Entity coordinates
    entity_coords_rad = np.radians(
        entity_df[
            ["latitude", "longitude"]
        ].values
    )

    # BallTree
    tree = BallTree(
        entity_coords_rad,
        metric="haversine"
    )

    # ------------------------------------------------
    # Nearest distance
    # ------------------------------------------------

    distances, _ = tree.query(
        event_coords_rad,
        k=1
    )

    result[
        f"distance_to_{entity_name}_km"
    ] = distances[:, 0] * EARTH_RADIUS_KM

    # ------------------------------------------------
    # 1 km count
    # ------------------------------------------------

    radius_1km_rad = radius_1km / EARTH_RADIUS_KM

    neighbours_1km = tree.query_radius(
        event_coords_rad,
        r=radius_1km_rad
    )

    result[
        f"{entity_name}_count_1km"
    ] = [
        len(x)
        for x in neighbours_1km
    ]

    # ------------------------------------------------
    # 3 km count
    # ------------------------------------------------

    radius_3km_rad = radius_3km / EARTH_RADIUS_KM

    neighbours_3km = tree.query_radius(
        event_coords_rad,
        r=radius_3km_rad
    )

    result[
        f"{entity_name}_count_3km"
    ] = [
        len(x)
        for x in neighbours_3km
    ]

    return result


# ------------------------------------------------
# Entity classes
# ------------------------------------------------

entity_classes = [
    "industrial_zone",
    "factory",
    "mine",
    "brick_industry",
    "works",
    "depot",
    "power_plant",
    "other_industry"
]

# ------------------------------------------------
# Calculate context
# ------------------------------------------------

osm_context = pd.DataFrame(
    index=events.index
)

for entity_class in entity_classes:

    entity_df = osm[
        osm["entity_class"] == entity_class
    ].copy()

    print(
        f"{entity_class:20s}: "
        f"{len(entity_df):5d} entities"
    )

    context = calculate_entity_context(
        entity_df,
        entity_class,
        event_coords_rad
    )

    osm_context = pd.concat(
        [osm_context, context],
        axis=1
    )


# ------------------------------------------------
# Merge into events
# ------------------------------------------------

osm_context = osm_context.reset_index(drop=True)

events = events.reset_index(drop=True)

events = pd.concat(
    [events, osm_context],
    axis=1
)

print("\n" + "=" * 70)
print("OSM CONTEXT COMPLETE")
print("=" * 70)

print(
    "Event table shape:",
    events.shape
)

CALCULATING EVENT → OSM ENTITY CONTEXT
industrial_zone     :   140 entities
factory             :    52 entities
mine                :     1 entities
brick_industry      :     6 entities
works               :  1563 entities
depot               :    39 entities
power_plant         :     0 entities
other_industry      :   171 entities

OSM CONTEXT COMPLETE
Event table shape: (3337, 49)


In [11]:
# ================================================================
# CELL 5E — OSM CONTEXT DIAGNOSTICS
# ================================================================

print("=" * 70)
print("OSM CONTEXT DIAGNOSTICS")
print("=" * 70)

distance_cols = [
    c for c in events.columns
    if c.startswith("distance_to_")
]

count_cols = [
    c for c in events.columns
    if "_count_" in c
]

print("\nDistance statistics:")
display(
    events[distance_cols]
    .replace(np.inf, np.nan)
    .describe()
    .T[
        ["count", "mean", "50%", "min", "max"]
    ]
)

print("\n" + "-" * 70)
print("EVENTS WITH ENTITY WITHIN 1 KM")
print("-" * 70)

for col in count_cols:

    count = (
        events[col] > 0
    ).sum()

    print(
        f"{col:40s}: "
        f"{count:5d} "
        f"({count / len(events) * 100:.2f}%)"
    )

print("\n" + "-" * 70)
print("EVENTS WITH ENTITY WITHIN 3 KM")
print("-" * 70)

for col in count_cols:

    col_3km = col.replace(
        "_count_1km",
        "_count_3km"
    )

    if col_3km in events.columns:

        count = (
            events[col_3km] > 0
        ).sum()

        print(
            f"{col_3km:40s}: "
            f"{count:5d} "
            f"({count / len(events) * 100:.2f}%)"
        )

OSM CONTEXT DIAGNOSTICS

Distance statistics:


,count,mean,50%,min,max
distance_to_industrial_zone_km,3337.0000,237.4009,199.3428,6.4681,1371.7452
distance_to_factory_km,3337.0000,369.2510,305.6822,6.7401,1484.7132
distance_to_mine_km,3337.0000,1365.5895,1405.7903,115.4637,2343.6984
distance_to_brick_industry_km,3337.0000,491.7328,421.2294,5.5978,2001.1049
distance_to_works_km,3337.0000,55.5132,49.1252,0.1298,1167.4031
distance_to_depot_km,3337.0000,325.2467,290.1604,2.2159,1235.5833
distance_to_power_plant_km,0.0000,NaN,NaN,NaN,NaN
distance_to_other_industry_km,3337.0000,267.1401,237.3644,2.0773,1222.0318



----------------------------------------------------------------------
EVENTS WITH ENTITY WITHIN 1 KM
----------------------------------------------------------------------
industrial_zone_count_1km               :     0 (0.00%)
industrial_zone_count_3km               :     0 (0.00%)
factory_count_1km                       :     0 (0.00%)
factory_count_3km                       :     0 (0.00%)
mine_count_1km                          :     0 (0.00%)
mine_count_3km                          :     0 (0.00%)
brick_industry_count_1km                :     0 (0.00%)
brick_industry_count_3km                :     0 (0.00%)
works_count_1km                         :    10 (0.30%)
works_count_3km                         :    64 (1.92%)
depot_count_1km                         :     0 (0.00%)
depot_count_3km                         :     1 (0.03%)
power_plant_count_1km                   :     0 (0.00%)
power_plant_count_3km                   :     0 (0.00%)
other_industry_count_1km                : 

In [12]:
# ================================================================
# CELL 6 — BEHAVIOR-ONLY K-MEANS
# ================================================================

print("=" * 70)
print("BEHAVIOR-ONLY K-MEANS CLUSTERING")
print("=" * 70)

# ------------------------------------------------
# Behavioral features
# ------------------------------------------------

behavior_features = [
    "mean_frp",
    "max_frp",
    "mean_bright_ti4",
    "max_bright_ti4",
    "mean_bright_ti5",
    "max_bright_ti5",
    "active_days",
    "duration_days",
    "activity_frequency",
    "detections_per_active_day",
    "spatial_extent_km"
]

print("Behavior features:")
for feature in behavior_features:
    print(" -", feature)

# ------------------------------------------------
# Prepare matrix
# ------------------------------------------------

X_behavior = events[
    behavior_features
].copy()

# Safety checks
print("\nMissing values:", X_behavior.isna().sum().sum())
print("Infinite values:", np.isinf(X_behavior).sum().sum())

# Replace infinities if any
X_behavior = X_behavior.replace(
    [np.inf, -np.inf],
    np.nan
)

# Median imputation as safety
X_behavior = X_behavior.fillna(
    X_behavior.median()
)

# ------------------------------------------------
# Robust scaling
# ------------------------------------------------

scaler_behavior = RobustScaler()

X_behavior_scaled = scaler_behavior.fit_transform(
    X_behavior
)

print("\nScaled matrix shape:")
print(X_behavior_scaled.shape)


# ------------------------------------------------
# Test K values
# ------------------------------------------------

results = []

for k in range(2, 7):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(
        X_behavior_scaled
    )

    silhouette = silhouette_score(
        X_behavior_scaled,
        labels
    )

    results.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette": silhouette
    })

results_df = pd.DataFrame(results)

print("\nK-Means evaluation:")
display(results_df)


# ------------------------------------------------
# Select K = 2
# ------------------------------------------------

best_k = 2

behavior_model = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=20
)

events["behavior_cluster"] = (
    behavior_model.fit_predict(
        X_behavior_scaled
    )
)

print("\n" + "=" * 70)
print("BEHAVIOR CLUSTERING COMPLETE")
print("=" * 70)

print(
    events["behavior_cluster"]
    .value_counts()
    .sort_index()
)

BEHAVIOR-ONLY K-MEANS CLUSTERING
Behavior features:
 - mean_frp
 - max_frp
 - mean_bright_ti4
 - max_bright_ti4
 - mean_bright_ti5
 - max_bright_ti5
 - active_days
 - duration_days
 - activity_frequency
 - detections_per_active_day
 - spatial_extent_km

Missing values: 0
Infinite values: 0

Scaled matrix shape:
(3337, 11)

K-Means evaluation:


,k,inertia,silhouette
0,2,53304.9271,0.8244
1,3,38605.2472,0.7121
2,4,29730.4914,0.5081
3,5,24265.0269,0.5227
4,6,21454.6369,0.4662



BEHAVIOR CLUSTERING COMPLETE
behavior_cluster
0    3195
1     142
Name: count, dtype: int64


In [13]:
# ================================================================
# CELL 6B — BEHAVIOR CLUSTER PROFILING
# ================================================================

print("=" * 70)
print("BEHAVIOR CLUSTER PROFILES")
print("=" * 70)

cluster_profile = (
    events
    .groupby("behavior_cluster")[behavior_features]
    .mean()
)

display(
    cluster_profile.round(4)
)

print("\nCluster sizes:")
display(
    events["behavior_cluster"]
    .value_counts()
    .sort_index()
    .rename("event_count")
)

print("\nCluster percentages:")
display(
    (
        events["behavior_cluster"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
        .rename("percentage")
    )
)

BEHAVIOR CLUSTER PROFILES


,mean_frp,max_frp,mean_bright_ti4,max_bright_ti4,mean_bright_ti5,max_bright_ti5,active_days,duration_days,activity_frequency,detections_per_active_day,spatial_extent_km
behavior_cluster,,,,,,,,,,,
0,1.1618,1.3540,304.2706,306.6469,283.6827,284.1003,1.3499,1.5189,0.9716,1.4846,0.1195
1,1.3649,3.2357,304.8101,321.0508,287.3448,291.1665,16.3099,20.1901,0.8092,2.8369,0.8284



Cluster sizes:


behavior_cluster
0    3195
1     142
Name: event_count, dtype: int64


Cluster percentages:


behavior_cluster
0   95.7400
1    4.2600
Name: percentage, dtype: float64

In [14]:
# ================================================================
# CELL 6C — PERSISTENCE PROFILE BY CLUSTER
# ================================================================

print("=" * 70)
print("PERSISTENCE PROFILE")
print("=" * 70)

persistence_profile = (
    events
    .groupby("behavior_cluster")
    .agg(
        events=("event_id", "count"),
        median_active_days=("active_days", "median"),
        mean_active_days=("active_days", "mean"),
        max_active_days=("active_days", "max"),
        median_duration_days=("duration_days", "median"),
        mean_duration_days=("duration_days", "mean"),
        max_duration_days=("duration_days", "max"),
        median_max_frp=("max_frp", "median"),
        median_spatial_extent=("spatial_extent_km", "median")
    )
)

display(
    persistence_profile.round(3)
)

PERSISTENCE PROFILE


,events,median_active_days,mean_active_days,max_active_days,median_duration_days,mean_duration_days,max_duration_days,median_max_frp,median_spatial_extent
behavior_cluster,,,,,,,,,
0,3195,1.0000,1.3500,9,1.0000,1.5190,13,1.0000,0.0000
1,142,15.0000,16.3100,31,17.0000,20.1900,31,2.6600,0.6280


In [15]:
# ================================================================
# CELL 7 — ASSIGN BEHAVIOR SEMANTICS
# ================================================================

print("=" * 70)
print("ASSIGNING BEHAVIOR SEMANTICS")
print("=" * 70)

# ------------------------------------------------
# Identify persistent cluster automatically
# ------------------------------------------------

cluster_active_days = (
    events
    .groupby("behavior_cluster")["active_days"]
    .median()
)

persistent_cluster = (
    cluster_active_days.idxmax()
)

transient_cluster = (
    cluster_active_days.idxmin()
)

print(
    f"Persistent-like cluster : "
    f"{persistent_cluster}"
)

print(
    f"Transient cluster        : "
    f"{transient_cluster}"
)

# ------------------------------------------------
# Semantic label
# ------------------------------------------------

events["behavior_type"] = np.where(
    events["behavior_cluster"] == persistent_cluster,
    "persistent_like",
    "transient"
)

# ------------------------------------------------
# Validation
# ------------------------------------------------

print("\nBehavior distribution:")

display(
    events["behavior_type"]
    .value_counts()
)

print("\nBehavior profile:")

display(
    events
    .groupby("behavior_type")
    [
        [
            "active_days",
            "duration_days",
            "max_frp",
            "spatial_extent_km"
        ]
    ]
    .agg(["count", "median", "mean"])
    .round(3)
)

ASSIGNING BEHAVIOR SEMANTICS
Persistent-like cluster : 1
Transient cluster        : 0

Behavior distribution:


behavior_type
transient          3195
persistent_like     142
Name: count, dtype: int64


Behavior profile:


active_days                 duration_days                  \
                      count  median    mean         count  median    mean   
behavior_type                                                               
persistent_like         142 15.0000 16.3100           142 17.0000 20.1900   
transient              3195  1.0000  1.3500          3195  1.0000  1.5190   

                max_frp               spatial_extent_km                
                  count median   mean             count median   mean  
behavior_type                                                          
persistent_like     142 2.6600 3.2360               142 0.6280 0.8280  
transient          3195 1.0000 1.3540              3195 0.0000 0.1200

In [16]:
# ================================================================
# CELL 8 — GEOGRAPHIC EVIDENCE FEATURES
# ================================================================

print("=" * 70)
print("BUILDING GEOGRAPHIC EVIDENCE FEATURES")
print("=" * 70)

# ------------------------------------------------
# WorldCover evidence
# ------------------------------------------------

events["is_builtup"] = (
    events["worldcover_class"]
    == "builtup"
).astype(int)

events["is_cropland"] = (
    events["worldcover_class"]
    == "crop"
).astype(int)

events["is_tree_cover"] = (
    events["worldcover_class"]
    == "tree"
).astype(int)

events["is_grassland"] = (
    events["worldcover_class"]
    == "grass"
).astype(int)

events["is_bare"] = (
    events["worldcover_class"]
    == "bare"
).astype(int)

events["is_shrubland"] = (
    events["worldcover_class"]
    == "shrub"
).astype(int)

events["is_water"] = (
    events["worldcover_class"]
    == "water"
).astype(int)


# ------------------------------------------------
# OSM proximity indicators
# ------------------------------------------------

events["near_works_1km"] = (
    events["works_count_1km"] > 0
).astype(int)

events["near_works_3km"] = (
    events["works_count_3km"] > 0
).astype(int)

events["near_industrial_zone_3km"] = (
    events["industrial_zone_count_3km"] > 0
).astype(int)

events["near_factory_3km"] = (
    events["factory_count_3km"] > 0
).astype(int)

events["near_depot_3km"] = (
    events["depot_count_3km"] > 0
).astype(int)

events["near_other_industry_3km"] = (
    events["other_industry_count_3km"] > 0
).astype(int)

events["near_mine_3km"] = (
    events["mine_count_3km"] > 0
).astype(int)

events["near_brick_industry_3km"] = (
    events["brick_industry_count_3km"] > 0
).astype(int)


# ------------------------------------------------
# Combined OSM evidence
# ------------------------------------------------

osm_evidence_cols = [
    "near_works_3km",
    "near_industrial_zone_3km",
    "near_factory_3km",
    "near_depot_3km",
    "near_other_industry_3km",
    "near_mine_3km",
    "near_brick_industry_3km"
]

events["osm_industrial_evidence"] = (
    events[osm_evidence_cols]
    .sum(axis=1)
)

# ------------------------------------------------
# Combined contextual flags
# ------------------------------------------------

events["strong_industrial_osm"] = (
    (
        events["near_factory_3km"] == 1
    )
    |
    (
        events["near_industrial_zone_3km"] == 1
    )
    |
    (
        events["near_brick_industry_3km"] == 1
    )
    |
    (
        events["near_mine_3km"] == 1
    )
).astype(int)


# ------------------------------------------------
# Summary
# ------------------------------------------------

print("WorldCover distribution:")

display(
    events["worldcover_class"]
    .value_counts(dropna=False)
)

print("\nOSM industrial evidence:")

display(
    events["osm_industrial_evidence"]
    .value_counts()
    .sort_index()
)

print("\nStrong industrial OSM evidence:")
print(
    events["strong_industrial_osm"].sum()
)

BUILDING GEOGRAPHIC EVIDENCE FEATURES
WorldCover distribution:


worldcover_class
tree       930
crop       734
grass      584
builtup    416
bare       331
NaN        195
shrub      109
water       37
wetland      1
Name: count, dtype: int64


OSM industrial evidence:


osm_industrial_evidence
0    3273
1      61
2       3
Name: count, dtype: int64


Strong industrial OSM evidence:
0


In [17]:
# ================================================================
# CELL 9 — DOMAIN EVIDENCE SCORE
# ================================================================

print("=" * 70)
print("BUILDING DOMAIN EVIDENCE SCORE")
print("=" * 70)

# ------------------------------------------------
# 1. BEHAVIOR EVIDENCE
# ------------------------------------------------

# Persistent-like behavior gets positive evidence.
events["behavior_score"] = np.where(
    events["behavior_type"] == "persistent_like",
    3.0,
    0.0
)


# ------------------------------------------------
# 2. THERMAL EVIDENCE
# ------------------------------------------------

# Use percentile ranks instead of arbitrary absolute thresholds.
# This makes the score relative to the current VIIRS dataset.

events["frp_percentile"] = (
    events["max_frp"]
    .rank(pct=True)
)

events["ti4_percentile"] = (
    events["max_bright_ti4"]
    .rank(pct=True)
)

events["thermal_score"] = (
    2.0 * events["frp_percentile"]
    +
    2.0 * events["ti4_percentile"]
)


# ------------------------------------------------
# 3. BUILT-UP CONTEXT
# ------------------------------------------------

# Built-up centroid is supporting evidence.
events["builtup_score"] = np.where(
    events["worldcover_class"] == "builtup",
    2.0,
    0.0
)


# ------------------------------------------------
# 4. AGRICULTURAL / NATURAL CONTEXT
# ------------------------------------------------

# These are not proof of non-industrial activity.
# They provide alternative-source evidence.

events["alternative_landcover_score"] = 0.0

events.loc[
    events["worldcover_class"] == "crop",
    "alternative_landcover_score"
] = 2.0

events.loc[
    events["worldcover_class"] == "tree",
    "alternative_landcover_score"
] = 1.5

events.loc[
    events["worldcover_class"] == "grass",
    "alternative_landcover_score"
] = 1.0

events.loc[
    events["worldcover_class"] == "water",
    "alternative_landcover_score"
] = 1.0


# ------------------------------------------------
# 5. OSM INDUSTRIAL EVIDENCE
# ------------------------------------------------

# OSM contributes ONLY when evidence exists.
# Absence does not subtract points.

events["osm_score"] = (
    events["near_works_3km"] * 2.0
    +
    events["near_industrial_zone_3km"] * 3.0
    +
    events["near_factory_3km"] * 3.0
    +
    events["near_depot_3km"] * 1.0
    +
    events["near_other_industry_3km"] * 2.0
    +
    events["near_mine_3km"] * 1.0
    +
    events["near_brick_industry_3km"] * 2.0
)


# ------------------------------------------------
# 6. FINAL INDUSTRIAL EVIDENCE SCORE
# ------------------------------------------------

events["industrial_evidence_score"] = (
    events["behavior_score"]
    +
    events["thermal_score"]
    +
    events["builtup_score"]
    +
    events["osm_score"]
    -
    events["alternative_landcover_score"]
)


# ------------------------------------------------
# Inspect score distribution
# ------------------------------------------------

print("\nScore statistics:")

display(
    events["industrial_evidence_score"]
    .describe()
    .round(3)
)

print("\nScore quantiles:")

display(
    events["industrial_evidence_score"]
    .quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99]
    )
    .round(3)
)

BUILDING DOMAIN EVIDENCE SCORE

Score statistics:


count   3337.0000
mean       1.3730
std        1.9920
min       -1.9820
25%       -0.0950
50%        1.1070
75%        2.4180
max       10.8910
Name: industrial_evidence_score, dtype: float64


Score quantiles:


0.5000   1.1070
0.7500   2.4180
0.9000   3.8600
0.9500   5.2420
0.9900   8.0010
Name: industrial_evidence_score, dtype: float64

In [18]:
# ================================================================
# CELL 10 — INDUSTRIAL CANDIDATE ANALYSIS
# ================================================================

print("=" * 70)
print("INDUSTRIAL CANDIDATE ANALYSIS")
print("=" * 70)

# ------------------------------------------------
# Rank all events
# ------------------------------------------------

candidate_columns = [
    "event_id",
    "behavior_type",
    "industrial_evidence_score",

    # Thermal
    "mean_frp",
    "max_frp",
    "max_bright_ti4",

    # Temporal
    "active_days",
    "duration_days",
    "detections_per_active_day",

    # Spatial
    "spatial_extent_km",

    # WorldCover
    "worldcover_class",

    # OSM
    "distance_to_works_km",
    "works_count_3km",
    "osm_industrial_evidence"
]

candidate_table = (
    events[
        candidate_columns
    ]
    .sort_values(
        "industrial_evidence_score",
        ascending=False
    )
)

print("\nTop 50 candidates:")

display(
    candidate_table
    .head(50)
    .round(3)
)


INDUSTRIAL CANDIDATE ANALYSIS

Top 50 candidates:


,event_id,behavior_type,industrial_evidence_score,mean_frp,max_frp,max_bright_ti4,active_days,duration_days,detections_per_active_day,spatial_extent_km,worldcover_class,distance_to_works_km,works_count_3km,osm_industrial_evidence
144,145,persistent_like,10.8910,1.6450,5.2100,333.3400,16,16,7.1250,1.2030,builtup,2.0580,2,1
37,38,persistent_like,10.8200,1.4750,4.0500,330.2500,24,31,1.9170,1.4450,builtup,1.2910,1,1
2041,2042,persistent_like,10.8170,1.6490,3.6900,332.0300,10,12,6.2000,1.1180,builtup,2.1590,2,1
842,843,persistent_like,10.2250,0.8130,1.6100,321.0000,15,21,1.6000,0.6310,builtup,1.6290,5,1
169,170,persistent_like,8.9780,2.2940,21.3700,344.2300,27,31,3.4810,1.4580,builtup,59.1370,0,0
28,29,persistent_like,8.9740,2.2270,8.9800,346.7600,17,18,7.7650,1.7370,builtup,41.8000,0,0
11,12,persistent_like,8.8610,1.8420,4.1100,335.2700,12,13,3.2500,1.3380,builtup,44.1270,0,0
125,126,persistent_like,8.8580,1.4980,4.0400,335.2200,14,14,4.7140,1.3720,builtup,28.4580,0,0
172,173,persistent_like,8.8570,1.4960,4.6200,331.5600,23,29,4.2610,1.4090,builtup,130.7570,0,0
2159,2160,persistent_like,8.8330,1.7920,3.8500,332.7600,8,11,6.1250,1.1440,builtup,37.2860,0,0


In [19]:
# ================================================================
# CELL 10B — SCORE BY BEHAVIOR TYPE
# ================================================================

print("=" * 70)
print("SCORE DISTRIBUTION BY BEHAVIOR")
print("=" * 70)

score_profile = (
    events
    .groupby("behavior_type")
    ["industrial_evidence_score"]
    .agg(
        count="count",
        minimum="min",
        median="median",
        mean="mean",
        p75=lambda x: x.quantile(0.75),
        p90=lambda x: x.quantile(0.90),
        p95=lambda x: x.quantile(0.95),
        maximum="max"
    )
)

display(
    score_profile.round(3)
)

SCORE DISTRIBUTION BY BEHAVIOR


,count,minimum,median,mean,p75,p90,p95,maximum
behavior_type,,,,,,,,
persistent_like,142,2.6250,6.6580,6.6540,7.7770,8.5720,8.8580,10.8910
transient,3195,-1.9820,0.9880,1.1390,2.2230,3.3510,4.0060,8.6860


In [20]:
# ================================================================
# CELL 10C — PERSISTENT CANDIDATE SCORE DISTRIBUTION
# ================================================================

print("=" * 70)
print("PERSISTENT-LIKE CANDIDATES")
print("=" * 70)

persistent_events = events[
    events["behavior_type"] == "persistent_like"
].copy()

print(
    f"Persistent-like events: "
    f"{len(persistent_events)}"
)

print("\nScore statistics:")

display(
    persistent_events[
        "industrial_evidence_score"
    ]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95
        ]
    )
    .round(3)
)

print("\nTop persistent candidates:")

display(
    persistent_events[
        candidate_columns
    ]
    .sort_values(
        "industrial_evidence_score",
        ascending=False
    )
    .head(30)
    .round(3)
)

PERSISTENT-LIKE CANDIDATES
Persistent-like events: 142

Score statistics:


count   142.0000
mean      6.6540
std       1.5310
min       2.6250
25%       5.6540
50%       6.6580
75%       7.7770
90%       8.5720
95%       8.8580
max      10.8910
Name: industrial_evidence_score, dtype: float64


Top persistent candidates:


,event_id,behavior_type,industrial_evidence_score,mean_frp,max_frp,max_bright_ti4,active_days,duration_days,detections_per_active_day,spatial_extent_km,worldcover_class,distance_to_works_km,works_count_3km,osm_industrial_evidence
144,145,persistent_like,10.8910,1.6450,5.2100,333.3400,16,16,7.1250,1.2030,builtup,2.0580,2,1
37,38,persistent_like,10.8200,1.4750,4.0500,330.2500,24,31,1.9170,1.4450,builtup,1.2910,1,1
2041,2042,persistent_like,10.8170,1.6490,3.6900,332.0300,10,12,6.2000,1.1180,builtup,2.1590,2,1
842,843,persistent_like,10.2250,0.8130,1.6100,321.0000,15,21,1.6000,0.6310,builtup,1.6290,5,1
169,170,persistent_like,8.9780,2.2940,21.3700,344.2300,27,31,3.4810,1.4580,builtup,59.1370,0,0
28,29,persistent_like,8.9740,2.2270,8.9800,346.7600,17,18,7.7650,1.7370,builtup,41.8000,0,0
11,12,persistent_like,8.8610,1.8420,4.1100,335.2700,12,13,3.2500,1.3380,builtup,44.1270,0,0
125,126,persistent_like,8.8580,1.4980,4.0400,335.2200,14,14,4.7140,1.3720,builtup,28.4580,0,0
172,173,persistent_like,8.8570,1.4960,4.6200,331.5600,23,29,4.2610,1.4090,builtup,130.7570,0,0
2159,2160,persistent_like,8.8330,1.7920,3.8500,332.7600,8,11,6.1250,1.1440,builtup,37.2860,0,0


In [21]:
# ================================================================
# CELL 9 — REVISED DOMAIN EVIDENCE MODEL
# ================================================================

print("=" * 70)
print("BUILDING REVISED DOMAIN EVIDENCE MODEL")
print("=" * 70)

# ------------------------------------------------
# 1. BEHAVIOR STRENGTH
# ------------------------------------------------

events["behavior_strength"] = np.where(
    events["behavior_type"] == "persistent_like",
    1.0,
    0.0
)


# ------------------------------------------------
# 2. THERMAL STRENGTH
# ------------------------------------------------

events["frp_percentile"] = (
    events["max_frp"]
    .rank(pct=True)
)

events["ti4_percentile"] = (
    events["max_bright_ti4"]
    .rank(pct=True)
)

events["thermal_strength"] = (
    events["frp_percentile"]
    +
    events["ti4_percentile"]
) / 2


# ------------------------------------------------
# 3. THERMAL-BEHAVIOR SCORE
# ------------------------------------------------

events["thermal_behavior_score"] = (
    0.5 * events["behavior_strength"]
    +
    0.5 * events["thermal_strength"]
)


# ------------------------------------------------
# 4. INDUSTRIAL CONTEXT
# ------------------------------------------------

# OSM is positive evidence ONLY when present.

events["osm_context_score"] = (
    0.50 * events["near_works_3km"]
    +
    0.25 * events["near_industrial_zone_3km"]
    +
    0.10 * events["near_factory_3km"]
    +
    0.05 * events["near_depot_3km"]
    +
    0.10 * events["near_other_industry_3km"]
)

# Built-up land is supporting context, NOT proof.
events["builtup_context_score"] = np.where(
    events["worldcover_class"] == "builtup",
    0.30,
    0.0
)

events["industrial_context_score"] = (
    events["osm_context_score"]
    +
    events["builtup_context_score"]
)


# ------------------------------------------------
# 5. ALTERNATIVE SOURCE CONTEXT
# ------------------------------------------------

events["alternative_context_score"] = 0.0

events.loc[
    events["worldcover_class"] == "crop",
    "alternative_context_score"
] = 0.30

events.loc[
    events["worldcover_class"] == "tree",
    "alternative_context_score"
] = 0.20

events.loc[
    events["worldcover_class"] == "grass",
    "alternative_context_score"
] = 0.15

events.loc[
    events["worldcover_class"] == "water",
    "alternative_context_score"
] = 0.20


# ------------------------------------------------
# 6. FINAL EVIDENCE SCORE
# ------------------------------------------------

events["final_evidence_score"] = (
    0.55 * events["thermal_behavior_score"]
    +
    0.35 * events["industrial_context_score"]
    -
    0.10 * events["alternative_context_score"]
)


# ------------------------------------------------
# Summary
# ------------------------------------------------

print("\nThermal-behavior score:")
display(
    events["thermal_behavior_score"]
    .describe()
    .round(3)
)

print("\nIndustrial context score:")
display(
    events["industrial_context_score"]
    .describe()
    .round(3)
)

print("\nAlternative context score:")
display(
    events["alternative_context_score"]
    .describe()
    .round(3)
)

print("\nFinal evidence score:")
display(
    events["final_evidence_score"]
    .describe()
    .round(3)
)

BUILDING REVISED DOMAIN EVIDENCE MODEL

Thermal-behavior score:


count   3337.0000
mean       0.2710
std        0.1880
min        0.0020
25%        0.1390
50%        0.2360
75%        0.3680
max        0.9990
Name: thermal_behavior_score, dtype: float64


Industrial context score:


count   3337.0000
mean       0.0470
std        0.1270
min        0.0000
25%        0.0000
50%        0.0000
75%        0.0000
max        0.9000
Name: industrial_context_score, dtype: float64


Alternative context score:


count   3337.0000
mean       0.1500
std        0.1130
min        0.0000
25%        0.0000
50%        0.2000
75%        0.2000
max        0.3000
Name: alternative_context_score, dtype: float64


Final evidence score:


count   3337.0000
mean       0.1510
std        0.1190
min       -0.0290
25%        0.0670
50%        0.1340
75%        0.2060
max        0.8230
Name: final_evidence_score, dtype: float64

In [23]:
print(events.columns.tolist())

['event_id', 'mean_frp', 'max_frp', 'std_frp', 'mean_bright_ti4', 'max_bright_ti4', 'std_bright_ti4', 'mean_bright_ti5', 'max_bright_ti5', 'std_bright_ti5', 'start_date', 'end_date', 'active_days', 'detection_count', 'duration_days', 'activity_frequency', 'detections_per_active_day', 'centroid_lat', 'centroid_lon', 'spatial_extent_km', 'frp_range', 'ti4_range', 'ti5_range', 'worldcover_class_code', 'worldcover_class', 'distance_to_industrial_zone_km', 'industrial_zone_count_1km', 'industrial_zone_count_3km', 'distance_to_factory_km', 'factory_count_1km', 'factory_count_3km', 'distance_to_mine_km', 'mine_count_1km', 'mine_count_3km', 'distance_to_brick_industry_km', 'brick_industry_count_1km', 'brick_industry_count_3km', 'distance_to_works_km', 'works_count_1km', 'works_count_3km', 'distance_to_depot_km', 'depot_count_1km', 'depot_count_3km', 'distance_to_power_plant_km', 'power_plant_count_1km', 'power_plant_count_3km', 'distance_to_other_industry_km', 'other_industry_count_1km', 'othe

In [27]:
print("=" * 70)
print("REVISED SCORE BY BEHAVIOR TYPE")
print("=" * 70)

score_summary = (
    events
    .groupby("behavior_type")["final_evidence_score"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        p25=lambda x: x.quantile(0.25),
        p75=lambda x: x.quantile(0.75),
        p90=lambda x: x.quantile(0.90),
        p95=lambda x: x.quantile(0.95),
        min="min",
        max="max"
    )
    .round(3)
)

display(score_summary)


print("\n" + "=" * 70)
print("TOP TRANSIENT EVENTS")
print("=" * 70)

transient_top = (
    events[events["behavior_type"] == "transient"]
    .sort_values("final_evidence_score", ascending=False)
    [[
        "event_id",
        "final_evidence_score",
        "thermal_behavior_score",
        "industrial_context_score",
        "alternative_context_score",
        "mean_frp",
        "max_frp",
        "max_bright_ti4",
        "active_days",
        "duration_days",
        "spatial_extent_km",
        "worldcover_class",
        "distance_to_works_km",
        "works_count_3km"
    ]]
    .head(15)
)

display(transient_top)


print("\n" + "=" * 70)
print("LOWEST-SCORING PERSISTENT EVENTS")
print("=" * 70)

persistent_low = (
    events[events["behavior_type"] == "persistent_like"]
    .sort_values("final_evidence_score", ascending=True)
    [[
        "event_id",
        "final_evidence_score",
        "thermal_behavior_score",
        "industrial_context_score",
        "alternative_context_score",
        "mean_frp",
        "max_frp",
        "max_bright_ti4",
        "active_days",
        "duration_days",
        "spatial_extent_km",
        "worldcover_class",
        "distance_to_works_km",
        "works_count_3km"
    ]]
    .head(15)
)

display(persistent_low)

REVISED SCORE BY BEHAVIOR TYPE


,count,mean,median,p25,p75,p90,p95,min,max
behavior_type,,,,,,,,,
persistent_like,142,0.5400,0.5270,0.5000,0.5800,0.6340,0.6450,0.3570,0.8230
transient,3195,0.1330,0.1270,0.0650,0.1960,0.2410,0.2640,-0.0290,0.5300



TOP TRANSIENT EVENTS


,event_id,final_evidence_score,thermal_behavior_score,industrial_context_score,alternative_context_score,mean_frp,max_frp,max_bright_ti4,active_days,duration_days,spatial_extent_km,worldcover_class,distance_to_works_km,works_count_3km
935,936,0.5298,0.4542,0.8000,0.0000,1.6167,2.8200,324.4900,3,5,0.2519,builtup,0.4199,1
2346,2347,0.5219,0.4398,0.8000,0.0000,1.6433,2.3900,322.0700,3,5,0.3134,builtup,2.9055,2
86,87,0.5053,0.4096,0.8000,0.0000,2.0360,3.0400,309.7700,5,7,0.1822,builtup,1.8823,1
1439,1440,0.5005,0.4008,0.8000,0.0000,2.1467,3.0300,308.5500,5,11,0.3795,builtup,1.2454,3
2108,2109,0.4997,0.3358,0.9000,0.0000,1.7900,1.7900,305.4200,1,1,0.0000,builtup,0.9723,6
2896,2897,0.4995,0.3992,0.8000,0.0000,3.5300,3.5300,307.3700,1,1,0.0000,builtup,1.9785,1
1842,1843,0.4987,0.3977,0.8000,0.0000,1.9275,2.3400,310.2800,4,7,0.2039,builtup,2.0633,1
1219,1220,0.4941,0.3892,0.8000,0.0000,3.3000,3.3000,306.6000,1,1,0.0000,builtup,2.1190,1
89,90,0.4768,0.3579,0.8000,0.0000,1.6450,2.1600,305.9600,4,7,0.1326,builtup,1.6744,1
1906,1907,0.4634,0.3335,0.8000,0.0000,1.4800,1.4800,307.0200,1,1,0.0000,builtup,1.5963,1



LOWEST-SCORING PERSISTENT EVENTS


,event_id,final_evidence_score,thermal_behavior_score,industrial_context_score,alternative_context_score,mean_frp,max_frp,max_bright_ti4,active_days,duration_days,spatial_extent_km,worldcover_class,distance_to_works_km,works_count_3km
682,683,0.3567,0.7031,0.0000,0.3000,0.4590,0.7000,304.7400,9,12,0.2795,crop,81.1853,0
265,266,0.3596,0.7083,0.0000,0.3000,0.3860,0.6700,305.6900,8,13,0.2284,crop,11.0629,0
181,182,0.3951,0.7183,0.0000,0.0000,0.7675,1.1700,300.4300,11,17,0.2226,bare,125.6814,0
1095,1096,0.4024,0.7861,0.0000,0.3000,0.6927,1.1900,305.2400,9,14,0.2560,crop,59.0235,0
109,110,0.4256,0.8283,0.0000,0.3000,0.6895,1.1900,310.2500,19,23,0.2332,crop,40.2090,0
725,726,0.4346,0.7901,0.0000,0.0000,0.7678,1.0400,307.5400,15,24,0.3031,bare,84.7892,0
219,220,0.4348,0.8451,0.0000,0.3000,0.6096,1.0900,317.5100,20,27,0.2710,crop,92.6803,0
957,958,0.4437,0.8430,0.0000,0.2000,0.7975,1.3800,309.3900,10,17,0.3254,tree,32.3199,0
142,143,0.4443,0.8078,0.0000,0.0000,0.8958,1.2000,307.1400,15,15,0.2984,bare,18.2408,0
1239,1240,0.4452,0.8094,0.0000,0.0000,0.8393,1.3800,305.5900,11,19,0.3337,bare,33.2473,0


In [28]:
print("=" * 70)
print("BEHAVIOR × INDUSTRIAL CONTEXT ANALYSIS")
print("=" * 70)

# Industrial context is considered present when any meaningful
# industrial OSM evidence exists within 3 km.
events["has_industrial_context"] = (
    events["industrial_context_score"] > 0
)

events["behavior_context_group"] = np.select(
    [
        (events["behavior_type"] == "persistent_like") &
        (events["has_industrial_context"]),

        (events["behavior_type"] == "persistent_like") &
        (~events["has_industrial_context"]),

        (events["behavior_type"] == "transient") &
        (events["has_industrial_context"]),

        (events["behavior_type"] == "transient") &
        (~events["has_industrial_context"])
    ],
    [
        "persistent + industrial",
        "persistent + no industrial",
        "transient + industrial",
        "transient + no industrial"
    ],
    default="other"
)


group_summary = (
    events
    .groupby("behavior_context_group")
    .agg(
        events=("event_id", "count"),
        mean_score=("final_evidence_score", "mean"),
        median_score=("final_evidence_score", "median"),
        p25_score=("final_evidence_score", lambda x: x.quantile(0.25)),
        p75_score=("final_evidence_score", lambda x: x.quantile(0.75)),
        p90_score=("final_evidence_score", lambda x: x.quantile(0.90)),
        mean_active_days=("active_days", "mean"),
        mean_max_frp=("max_frp", "mean"),
        mean_max_ti4=("max_bright_ti4", "mean")
    )
    .sort_values("median_score", ascending=False)
    .round(3)
)

display(group_summary)


print("\n" + "=" * 70)
print("COUNTS")
print("=" * 70)

display(
    events["behavior_context_group"]
    .value_counts()
    .rename_axis("group")
    .reset_index(name="events")
)


print("\n" + "=" * 70)
print("INDUSTRIAL-CONTEXT EVENTS")
print("=" * 70)

industrial_context_events = (
    events[events["has_industrial_context"]]
    .sort_values("final_evidence_score", ascending=False)
    [[
        "event_id",
        "behavior_type",
        "final_evidence_score",
        "thermal_behavior_score",
        "industrial_context_score",
        "alternative_context_score",
        "active_days",
        "duration_days",
        "mean_frp",
        "max_frp",
        "max_bright_ti4",
        "worldcover_class",
        "distance_to_works_km",
        "works_count_3km",
        "distance_to_industrial_zone_km",
        "industrial_zone_count_3km",
        "distance_to_factory_km",
        "factory_count_3km"
    ]]
)

display(industrial_context_events)

BEHAVIOR × INDUSTRIAL CONTEXT ANALYSIS


,events,mean_score,median_score,p25_score,p75_score,p90_score,mean_active_days,mean_max_frp,mean_max_ti4
behavior_context_group,,,,,,,,,
persistent + industrial,43,0.6270,0.6200,0.5940,0.6440,0.6590,14.8840,3.1060,318.6580
persistent + no industrial,99,0.5020,0.5080,0.4880,0.5300,0.5440,16.9290,3.2920,322.0900
transient + industrial,413,0.2410,0.2300,0.1750,0.2890,0.3430,1.8960,1.2010,304.7610
transient + no industrial,2782,0.1170,0.1090,0.0570,0.1800,0.2240,1.2690,1.3770,306.9270



COUNTS


,group,events
0,transient + no industrial,2782
1,transient + industrial,413
2,persistent + no industrial,99
3,persistent + industrial,43



INDUSTRIAL-CONTEXT EVENTS


,event_id,behavior_type,final_evidence_score,thermal_behavior_score,industrial_context_score,alternative_context_score,active_days,duration_days,mean_frp,max_frp,max_bright_ti4,worldcover_class,distance_to_works_km,works_count_3km,distance_to_industrial_zone_km,industrial_zone_count_3km,distance_to_factory_km,factory_count_3km
144,145,persistent_like,0.8225,0.9864,0.8000,0.0000,16,16,1.6448,5.2100,333.3400,builtup,2.0576,2,123.3338,0,244.3338,0
37,38,persistent_like,0.8176,0.9775,0.8000,0.0000,24,31,1.4754,4.0500,330.2500,builtup,1.2907,1,178.1249,0,227.1997,0
2041,2042,persistent_like,0.8174,0.9771,0.8000,0.0000,10,12,1.6494,3.6900,332.0300,builtup,2.1590,2,123.2699,0,244.3447,0
842,843,persistent_like,0.7767,0.9031,0.8000,0.0000,15,21,0.8133,1.6100,321.0000,builtup,1.6294,5,91.0162,0,749.8525,0
83,84,persistent_like,0.6598,0.8815,0.5000,0.0000,17,20,0.8696,1.6200,313.4200,bare,2.1640,1,244.4622,0,910.9798,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311,312,transient,0.1211,0.0293,0.3000,0.0000,1,1,0.4600,0.4600,295.4600,builtup,42.4252,0,111.8292,0,423.4404,0
1624,1625,transient,0.1202,0.0276,0.3000,0.0000,1,1,0.4300,0.4300,295.6500,builtup,17.7219,0,227.7153,0,204.1244,0
2130,2131,transient,0.1171,0.0220,0.3000,0.0000,1,1,0.3300,0.3300,296.0400,builtup,113.0331,0,226.4789,0,385.6015,0
568,569,transient,0.1110,0.0109,0.3000,0.0000,1,1,0.3100,0.3100,295.4200,builtup,23.3474,0,12.9685,0,367.2345,0


In [29]:
print("=" * 70)
print("OSM EVIDENCE BREAKDOWN")
print("=" * 70)

evidence_columns = [
    "near_works_3km",
    "near_industrial_zone_3km",
    "near_factory_3km",
    "near_depot_3km",
    "near_other_industry_3km",
    "near_mine_3km",
    "near_brick_industry_3km"
]

for col in evidence_columns:
    mask = events[col] == 1

    print(
        f"{col:<35} "
        f"{mask.sum():>5} events "
        f"({mask.mean()*100:>6.2f}%)"
    )


print("\n" + "=" * 70)
print("INDUSTRIAL CONTEXT SCORE DISTRIBUTION")
print("=" * 70)

display(
    events["industrial_context_score"]
    .value_counts()
    .sort_index()
    .to_frame("events")
)


print("\n" + "=" * 70)
print("PERSISTENT INDUSTRIAL EVENTS — OSM SIGNAL")
print("=" * 70)

persistent_industrial = (
    events[
        (events["behavior_type"] == "persistent_like") &
        (events["industrial_context_score"] > 0)
    ]
    [[
        "event_id",
        "final_evidence_score",
        "industrial_context_score",
        "near_works_3km",
        "near_industrial_zone_3km",
        "near_factory_3km",
        "near_depot_3km",
        "near_other_industry_3km",
        "near_mine_3km",
        "near_brick_industry_3km",
        "distance_to_works_km",
        "works_count_3km",
        "worldcover_class",
        "active_days",
        "max_frp",
        "max_bright_ti4"
    ]]
    .sort_values("final_evidence_score", ascending=False)
)

display(persistent_industrial)

OSM EVIDENCE BREAKDOWN
near_works_3km                         64 events (  1.92%)
near_industrial_zone_3km                0 events (  0.00%)
near_factory_3km                        0 events (  0.00%)
near_depot_3km                          1 events (  0.03%)
near_other_industry_3km                 2 events (  0.06%)
near_mine_3km                           0 events (  0.00%)
near_brick_industry_3km                 0 events (  0.00%)

INDUSTRIAL CONTEXT SCORE DISTRIBUTION


,events
industrial_context_score,
0.0000,2881
0.3000,392
0.5000,39
0.6000,1
0.8000,22
0.8500,1
0.9000,1



PERSISTENT INDUSTRIAL EVENTS — OSM SIGNAL


,event_id,final_evidence_score,industrial_context_score,near_works_3km,near_industrial_zone_3km,near_factory_3km,near_depot_3km,near_other_industry_3km,near_mine_3km,near_brick_industry_3km,distance_to_works_km,works_count_3km,worldcover_class,active_days,max_frp,max_bright_ti4
144,145,0.8225,0.8000,1,0,0,0,0,0,0,2.0576,2,builtup,16,5.2100,333.3400
37,38,0.8176,0.8000,1,0,0,0,0,0,0,1.2907,1,builtup,24,4.0500,330.2500
2041,2042,0.8174,0.8000,1,0,0,0,0,0,0,2.1590,2,builtup,10,3.6900,332.0300
842,843,0.7767,0.8000,1,0,0,0,0,0,0,1.6294,5,builtup,15,1.6100,321.0000
83,84,0.6598,0.5000,1,0,0,0,0,0,0,2.1640,1,bare,17,1.6200,313.4200
169,170,0.6535,0.3000,0,0,0,0,0,0,0,59.1371,0,builtup,27,21.3700,344.2300
28,29,0.6532,0.3000,0,0,0,0,0,0,0,41.7997,0,builtup,17,8.9800,346.7600
11,12,0.6454,0.3000,0,0,0,0,0,0,0,44.1268,0,builtup,12,4.1100,335.2700
125,126,0.6452,0.3000,0,0,0,0,0,0,0,28.4580,0,builtup,14,4.0400,335.2200
172,173,0.6452,0.3000,0,0,0,0,0,0,0,130.7572,0,builtup,23,4.6200,331.5600


In [30]:
print("=" * 70)
print("BUILD FINAL DOMAIN CLASSIFICATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Define evidence levels
# ------------------------------------------------------------

# Stronger industrial context:
# current OSM data is sparse, so works is the main available
# positive industrial signal.
events["has_osm_industrial"] = (
    events["industrial_context_score"] >= 0.5
)

# Moderate thermal evidence
events["strong_thermal"] = (
    events["thermal_behavior_score"] >= 0.50
)

# Strong thermal evidence
events["very_strong_thermal"] = (
    events["thermal_behavior_score"] >= 0.70
)

# Alternative environmental context
events["has_alternative_context"] = (
    events["alternative_context_score"] >= 0.20
)


# ------------------------------------------------------------
# 2. Final domain rules
# ------------------------------------------------------------

events["final_class"] = "Uncertain"


# HIGH-CONFIDENCE INDUSTRIAL
# Persistent thermal behavior + meaningful OSM industrial evidence
industrial_rule_1 = (
    (events["behavior_type"] == "persistent_like") &
    events["has_osm_industrial"] &
    (events["thermal_behavior_score"] >= 0.50)
)

# INDUSTRIAL FIRE CANDIDATE
# Transient/any behavior can still represent an industrial fire
# if thermal behavior and industrial geographic context are both strong.
industrial_rule_2 = (
    events["has_osm_industrial"] &
    events["very_strong_thermal"]
)

events.loc[
    industrial_rule_1 | industrial_rule_2,
    "final_class"
] = "Industrial"


# NON-INDUSTRIAL
# Weak thermal behavior + no industrial context +
# environmental alternative evidence.
nonindustrial_rule = (
    (events["thermal_behavior_score"] < 0.35) &
    (~events["has_osm_industrial"]) &
    events["has_alternative_context"]
)

events.loc[
    nonindustrial_rule,
    "final_class"
] = "Non-industrial"


# ------------------------------------------------------------
# 3. Classification summary
# ------------------------------------------------------------

print("\nFinal classification:")
print(
    events["final_class"]
    .value_counts()
)


print("\nPercentages:")
print(
    (events["final_class"].value_counts(normalize=True) * 100)
    .round(2)
)


# ------------------------------------------------------------
# 4. Classification by behavior
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASSIFICATION × BEHAVIOR")
print("=" * 70)

classification_behavior = pd.crosstab(
    events["behavior_type"],
    events["final_class"]
)

display(classification_behavior)


# ------------------------------------------------------------
# 5. Final candidate table
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INDUSTRIAL CANDIDATES")
print("=" * 70)

industrial_candidates = (
    events[events["final_class"] == "Industrial"]
    .sort_values("final_evidence_score", ascending=False)
    [[
        "event_id",
        "final_class",
        "behavior_type",
        "final_evidence_score",
        "thermal_behavior_score",
        "industrial_context_score",
        "alternative_context_score",
        "active_days",
        "duration_days",
        "mean_frp",
        "max_frp",
        "max_bright_ti4",
        "spatial_extent_km",
        "worldcover_class",
        "distance_to_works_km",
        "works_count_3km"
    ]]
)

display(industrial_candidates)


# ------------------------------------------------------------
# 6. Save final output
# ------------------------------------------------------------

events.to_csv(
    V10_FINAL_FILE,
    index=False
)

print(f"\nSaved: {V10_FINAL_FILE}")
print(f"Rows: {len(events):,}")
print(f"Columns: {len(events.columns):,}")

BUILD FINAL DOMAIN CLASSIFICATION

Final classification:
final_class
Uncertain         2067
Non-industrial    1263
Industrial           7
Name: count, dtype: int64

Percentages:
final_class
Uncertain        61.9400
Non-industrial   37.8500
Industrial        0.2100
Name: proportion, dtype: float64

CLASSIFICATION × BEHAVIOR


final_class,Industrial,Non-industrial,Uncertain
behavior_type,,,
persistent_like,7,0,135
transient,0,1263,1932



INDUSTRIAL CANDIDATES


,event_id,final_class,behavior_type,final_evidence_score,thermal_behavior_score,industrial_context_score,alternative_context_score,active_days,duration_days,mean_frp,max_frp,max_bright_ti4,spatial_extent_km,worldcover_class,distance_to_works_km,works_count_3km
144,145,Industrial,persistent_like,0.8225,0.9864,0.8000,0.0000,16,16,1.6448,5.2100,333.3400,1.2032,builtup,2.0576,2
37,38,Industrial,persistent_like,0.8176,0.9775,0.8000,0.0000,24,31,1.4754,4.0500,330.2500,1.4448,builtup,1.2907,1
2041,2042,Industrial,persistent_like,0.8174,0.9771,0.8000,0.0000,10,12,1.6494,3.6900,332.0300,1.1183,builtup,2.1590,2
842,843,Industrial,persistent_like,0.7767,0.9031,0.8000,0.0000,15,21,0.8133,1.6100,321.0000,0.6310,builtup,1.6294,5
83,84,Industrial,persistent_like,0.6598,0.8815,0.5000,0.0000,17,20,0.8696,1.6200,313.4200,0.9193,bare,2.1640,1
258,259,Industrial,persistent_like,0.6440,0.8527,0.5000,0.0000,20,30,1.0935,1.6400,308.0100,0.4694,bare,2.7398,1
1505,1506,Industrial,persistent_like,0.6339,0.8708,0.5000,0.2000,13,18,0.9475,1.8800,308.7100,1.0465,tree,0.3608,2



Saved: viirs_event_final_classification_v10.csv
Rows: 3,337
Columns: 91


In [31]:
print("=" * 70)
print("PERSISTENT-LIKE EVENT TIERS")
print("=" * 70)

persistent = events[
    events["behavior_type"] == "persistent_like"
].copy()

# Thermal strength tiers
persistent["thermal_tier"] = np.select(
    [
        persistent["thermal_behavior_score"] >= 0.80,
        persistent["thermal_behavior_score"] >= 0.65,
        persistent["thermal_behavior_score"] >= 0.50
    ],
    [
        "very_strong",
        "strong",
        "moderate"
    ],
    default="weak"
)

# Geographic support tiers
persistent["context_tier"] = np.select(
    [
        persistent["industrial_context_score"] >= 0.50,
        persistent["industrial_context_score"] > 0
    ],
    [
        "strong_industrial",
        "weak_industrial"
    ],
    default="none"
)

tier_summary = (
    persistent
    .groupby(["thermal_tier", "context_tier"])
    .agg(
        events=("event_id", "count"),
        mean_score=("final_evidence_score", "mean"),
        median_score=("final_evidence_score", "median"),
        mean_active_days=("active_days", "mean"),
        mean_max_frp=("max_frp", "mean"),
        mean_max_ti4=("max_bright_ti4", "mean")
    )
    .reset_index()
    .sort_values(
        ["thermal_tier", "context_tier"],
        ascending=[False, True]
    )
    .round(3)
)

display(tier_summary)


print("\n" + "=" * 70)
print("VERY STRONG PERSISTENT EVENTS WITHOUT OSM")
print("=" * 70)

display(
    persistent[
        (persistent["thermal_behavior_score"] >= 0.70) &
        (persistent["industrial_context_score"] == 0)
    ]
    [[
        "event_id",
        "thermal_behavior_score",
        "final_evidence_score",
        "active_days",
        "duration_days",
        "mean_frp",
        "max_frp",
        "max_bright_ti4",
        "spatial_extent_km",
        "worldcover_class",
        "alternative_context_score",
        "distance_to_works_km"
    ]]
    .sort_values("thermal_behavior_score", ascending=False)
)


print("\n" + "=" * 70)
print("STRONG TRANSIENT EVENTS WITH OSM")
print("=" * 70)

display(
    events[
        (events["behavior_type"] == "transient") &
        (events["industrial_context_score"] > 0) &
        (events["thermal_behavior_score"] >= 0.50)
    ]
    [[
        "event_id",
        "final_evidence_score",
        "thermal_behavior_score",
        "industrial_context_score",
        "active_days",
        "duration_days",
        "mean_frp",
        "max_frp",
        "max_bright_ti4",
        "spatial_extent_km",
        "worldcover_class",
        "distance_to_works_km",
        "works_count_3km"
    ]]
    .sort_values("thermal_behavior_score", ascending=False)
)

PERSISTENT-LIKE EVENT TIERS


,thermal_tier,context_tier,events,mean_score,median_score,mean_active_days,mean_max_frp,mean_max_ti4
2,very_strong,none,94,0.5080,0.5090,17.2770,3.4170,323.0140
3,very_strong,strong_industrial,7,0.7390,0.7770,16.4290,2.8140,320.9660
4,very_strong,weak_industrial,34,0.6100,0.6130,14.8530,3.2950,318.8520
0,strong,none,5,0.3900,0.3950,10.4000,0.9540,304.7280
1,strong,weak_industrial,2,0.5180,0.5180,10.0000,0.9050,307.2750



VERY STRONG PERSISTENT EVENTS WITHOUT OSM


,event_id,thermal_behavior_score,final_evidence_score,active_days,duration_days,mean_frp,max_frp,max_bright_ti4,spatial_extent_km,worldcover_class,alternative_context_score,distance_to_works_km
92,93,0.9992,0.5495,31,31,3.1842,15.6900,354.2000,1.4565,bare,0.0000,19.2346
55,56,0.9961,0.5479,26,31,2.0403,7.3400,348.6000,1.7617,bare,0.0000,31.3727
188,189,0.9961,0.5478,27,31,2.5160,6.3400,356.5200,1.5231,bare,0.0000,68.4492
193,194,0.9960,0.5478,20,21,2.5248,7.9600,347.0300,1.4236,bare,0.0000,41.9549
10,11,0.9948,0.5471,23,31,1.6210,5.5800,349.7500,1.1611,bare,0.0000,46.2580
108,109,0.9934,0.5464,26,31,1.9137,5.7000,344.2900,2.6213,bare,0.0000,26.6134
33,34,0.9918,0.5455,27,31,1.8416,5.1300,343.5600,2.8129,bare,0.0000,15.1887
127,128,0.9918,0.5455,26,29,1.8142,6.4600,339.0800,1.6447,bare,0.0000,40.5230
2148,2149,0.9917,0.5454,8,12,1.9018,9.5800,335.7200,0.3818,bare,0.0000,92.6900
1966,1967,0.9901,0.5246,10,13,2.1372,5.4800,338.0700,1.0054,tree,0.2000,28.5209



STRONG TRANSIENT EVENTS WITH OSM


,event_id,final_evidence_score,thermal_behavior_score,industrial_context_score,active_days,duration_days,mean_frp,max_frp,max_bright_ti4,spatial_extent_km,worldcover_class,distance_to_works_km,works_count_3km


In [32]:
# ================================================================
# FINAL DOMAIN CLASSIFICATION — V10
# ================================================================

print("=" * 70)
print("FINAL DOMAIN CLASSIFICATION")
print("=" * 70)

# ------------------------------------------------
# 1. Evidence flags
# ------------------------------------------------

events["has_industrial_context"] = (
    events["industrial_context_score"] >= 0.50
)

events["has_any_industrial_context"] = (
    events["industrial_context_score"] > 0
)

events["strong_thermal"] = (
    events["thermal_behavior_score"] >= 0.65
)

events["very_strong_thermal"] = (
    events["thermal_behavior_score"] >= 0.80
)

events["alternative_context"] = (
    events["alternative_context_score"] >= 0.20
)


# ------------------------------------------------
# 2. Start everything as Uncertain
# ------------------------------------------------

events["final_class"] = "Uncertain"


# ------------------------------------------------
# 3. HIGH-CONFIDENCE INDUSTRIAL
#
# Strong thermal behavior + strong industrial
# geographic evidence.
#
# Persistence is NOT mandatory because an
# industrial fire can be transient.
# ------------------------------------------------

industrial_rule = (
    events["strong_thermal"] &
    events["has_industrial_context"]
)

events.loc[
    industrial_rule,
    "final_class"
] = "Industrial"


# ------------------------------------------------
# 4. NON-INDUSTRIAL
#
# Conservative rule:
# weak thermal behavior + no industrial evidence
# + environmental/alternative context.
# ------------------------------------------------

nonindustrial_rule = (
    (events["thermal_behavior_score"] < 0.35) &
    (~events["has_any_industrial_context"]) &
    events["alternative_context"]
)

events.loc[
    nonindustrial_rule,
    "final_class"
] = "Non-industrial"


# ------------------------------------------------
# 5. Summary
# ------------------------------------------------

print("\nFinal classification:")
display(
    events["final_class"]
    .value_counts()
    .rename_axis("class")
    .to_frame("events")
)

print("\nPercentage:")
display(
    (
        events["final_class"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
        .rename_axis("class")
        .to_frame("percentage")
    )
)


# ------------------------------------------------
# 6. Behavior × final classification
# ------------------------------------------------

print("\n" + "=" * 70)
print("BEHAVIOR × FINAL CLASS")
print("=" * 70)

display(
    pd.crosstab(
        events["behavior_type"],
        events["final_class"]
    )
)


# ------------------------------------------------
# 7. Industrial candidates
# ------------------------------------------------

print("\n" + "=" * 70)
print("INDUSTRIAL CANDIDATES")
print("=" * 70)

industrial_candidates = (
    events[events["final_class"] == "Industrial"]
    .sort_values(
        ["final_evidence_score", "thermal_behavior_score"],
        ascending=False
    )
)

display(
    industrial_candidates[[
        "event_id",
        "final_class",
        "behavior_type",
        "final_evidence_score",
        "thermal_behavior_score",
        "industrial_context_score",
        "alternative_context_score",
        "active_days",
        "duration_days",
        "mean_frp",
        "max_frp",
        "max_bright_ti4",
        "spatial_extent_km",
        "centroid_lat",
        "centroid_lon",
        "worldcover_class",
        "distance_to_works_km",
        "works_count_3km"
    ]]
)


# ------------------------------------------------
# 8. GIS-ready output
# ------------------------------------------------

gis_columns = [
    "event_id",
    "centroid_lat",
    "centroid_lon",
    "final_class",
    "behavior_type",
    "final_evidence_score",
    "thermal_behavior_score",
    "industrial_context_score",
    "alternative_context_score",
    "active_days",
    "duration_days",
    "detection_count",
    "mean_frp",
    "max_frp",
    "mean_bright_ti4",
    "max_bright_ti4",
    "spatial_extent_km",
    "worldcover_class",
    "distance_to_works_km",
    "works_count_3km"
]

gis_events = events[gis_columns].copy()

gis_events.to_csv(
    "viirs_events_gis_v10.csv",
    index=False
)

# Full analytical table
events.to_csv(
    V10_FINAL_FILE,
    index=False
)

print("\nSaved:")
print("  viirs_event_final_classification_v10.csv")
print("  viirs_events_gis_v10.csv")

print("\nGIS dataset shape:", gis_events.shape)

FINAL DOMAIN CLASSIFICATION

Final classification:


,events
class,
Uncertain,2067
Non-industrial,1263
Industrial,7



Percentage:


,percentage
class,
Uncertain,61.9400
Non-industrial,37.8500
Industrial,0.2100



BEHAVIOR × FINAL CLASS


final_class,Industrial,Non-industrial,Uncertain
behavior_type,,,
persistent_like,7,0,135
transient,0,1263,1932



INDUSTRIAL CANDIDATES


,event_id,final_class,behavior_type,final_evidence_score,thermal_behavior_score,industrial_context_score,alternative_context_score,active_days,duration_days,mean_frp,max_frp,max_bright_ti4,spatial_extent_km,centroid_lat,centroid_lon,worldcover_class,distance_to_works_km,works_count_3km
144,145,Industrial,persistent_like,0.8225,0.9864,0.8000,0.0000,16,16,1.6448,5.2100,333.3400,1.2032,22.7906,86.2032,builtup,2.0576,2
37,38,Industrial,persistent_like,0.8176,0.9775,0.8000,0.0000,24,31,1.4754,4.0500,330.2500,1.4448,19.1005,82.1671,builtup,1.2907,1
2041,2042,Industrial,persistent_like,0.8174,0.9771,0.8000,0.0000,10,12,1.6494,3.6900,332.0300,1.1183,22.7914,86.2029,builtup,2.1590,2
842,843,Industrial,persistent_like,0.7767,0.9031,0.8000,0.0000,15,21,0.8133,1.6100,321.0000,0.6310,22.9785,72.5661,builtup,1.6294,5
83,84,Industrial,persistent_like,0.6598,0.8815,0.5000,0.0000,17,20,0.8696,1.6200,313.4200,0.9193,23.1145,70.0862,bare,2.1640,1
258,259,Industrial,persistent_like,0.6440,0.8527,0.5000,0.0000,20,30,1.0935,1.6400,308.0100,0.4694,20.8355,70.6932,bare,2.7398,1
1505,1506,Industrial,persistent_like,0.6339,0.8708,0.5000,0.2000,13,18,0.9475,1.8800,308.7100,1.0465,19.9221,79.1136,tree,0.3608,2



Saved:
  viirs_event_final_classification_v10.csv
  viirs_events_gis_v10.csv

GIS dataset shape: (3337, 20)


In [33]:
%pip install folium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\vinja\Projects\llm_engineering\.venv\Scripts\python.exe -m pip install --upgrade pip


In [34]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

GIS_FILE = "viirs_events_gis_v10.csv"

gis = pd.read_csv(GIS_FILE)

print("Shape:", gis.shape)
print("\nFinal classes:")
print(gis["final_class"].value_counts())

print("\nColumns:")
print(gis.columns.tolist())

Shape: (3337, 20)

Final classes:
final_class
Uncertain         2067
Non-industrial    1263
Industrial           7
Name: count, dtype: int64

Columns:
['event_id', 'centroid_lat', 'centroid_lon', 'final_class', 'behavior_type', 'final_evidence_score', 'thermal_behavior_score', 'industrial_context_score', 'alternative_context_score', 'active_days', 'duration_days', 'detection_count', 'mean_frp', 'max_frp', 'mean_bright_ti4', 'max_bright_ti4', 'spatial_extent_km', 'worldcover_class', 'distance_to_works_km', 'works_count_3km']


In [35]:
required_cols = [
    "event_id",
    "centroid_lat",
    "centroid_lon",
    "final_class",
    "behavior_type",
    "final_evidence_score",
    "thermal_behavior_score",
    "industrial_context_score",
    "active_days",
    "duration_days",
    "mean_frp",
    "max_frp",
    "max_bright_ti4"
]

missing = [c for c in required_cols if c not in gis.columns]

print("Missing required columns:", missing)

print("\nCoordinate range:")
print("Latitude :", gis["centroid_lat"].min(), "to", gis["centroid_lat"].max())
print("Longitude:", gis["centroid_lon"].min(), "to", gis["centroid_lon"].max())

print("\nMissing coordinates:")
print(gis[["centroid_lat", "centroid_lon"]].isna().sum())

Missing required columns: []

Coordinate range:
Latitude : 8.24339 to 34.62992
Longitude: 68.57577 to 97.07546

Missing coordinates:
centroid_lat    0
centroid_lon    0
dtype: int64


In [36]:
m = folium.Map(
    location=[22.5, 79.0],
    zoom_start=5,
    tiles="CartoDB positron"
)

print("Base map created.")

Base map created.


In [37]:
industrial_group = MarkerCluster(
    name="Industrial Candidates",
    overlay=True,
    control=True
)

nonindustrial_group = MarkerCluster(
    name="Non-industrial",
    overlay=True,
    control=True
)

uncertain_group = MarkerCluster(
    name="Uncertain",
    overlay=True,
    control=True
)

for _, row in gis.iterrows():

    lat = row["centroid_lat"]
    lon = row["centroid_lon"]

    if pd.isna(lat) or pd.isna(lon):
        continue

    popup = f"""
    <b>Event ID:</b> {int(row['event_id'])}<br>
    <b>Class:</b> {row['final_class']}<br>
    <b>Behavior:</b> {row['behavior_type']}<br>
    <b>Evidence Score:</b> {row['final_evidence_score']:.3f}<br>
    <b>Thermal Score:</b> {row['thermal_behavior_score']:.3f}<br>
    <b>Industrial Context:</b> {row['industrial_context_score']:.3f}<br>
    <b>Active Days:</b> {int(row['active_days'])}<br>
    <b>Duration:</b> {int(row['duration_days'])} days<br>
    <b>Mean FRP:</b> {row['mean_frp']:.2f}<br>
    <b>Max FRP:</b> {row['max_frp']:.2f}<br>
    <b>Max TI4:</b> {row['max_bright_ti4']:.2f} K
    """

    marker = folium.CircleMarker(
        location=[lat, lon],
        radius=4 if row["final_class"] != "Industrial" else 8,
        popup=folium.Popup(popup, max_width=300),
        fill=True,
        fill_opacity=0.7,
        weight=1
    )

    if row["final_class"] == "Industrial":
        marker.add_to(industrial_group)

    elif row["final_class"] == "Non-industrial":
        marker.add_to(nonindustrial_group)

    else:
        marker.add_to(uncertain_group)


industrial_group.add_to(m)
nonindustrial_group.add_to(m)
uncertain_group.add_to(m)

folium.LayerControl().add_to(m)

m

In [38]:
GIS_HTML = "viirs_india_gis_v10.html"

m.save(GIS_HTML)

print(f"Saved: {GIS_HTML}")

Saved: viirs_india_gis_v10.html
